In [1]:
!pip install earthengine-api geemap rasterio torch torchvision segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.9 MB/s eta 0:00:00


In [9]:
import ee
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Authenticate and Initialize Earth Engine
# This will pop up a window asking for your Google login
ee.Authenticate()
# Replace 'your-gcp-project-id' with your actual Google Cloud Project ID
# TODO: Replace 'your-gcp-project-id' with your actual Google Cloud Project ID
ee.Initialize(project='carbon-estimation-project')

# 3. Define the Region of Interest (ROI) - Rondônia, Brazil
roi = ee.Geometry.BBox(-62.5, -10.5, -62.0, -10.0)

# 4. Cloud Masking Function
def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000)

# 5. Fetch Input (X) - Sentinel-2 Optical
s2_image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi)
            .filterDate('2022-01-01', '2022-12-31')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
            .map(mask_s2_clouds)
            .median()
            .select(['B4', 'B3', 'B2'])
            .clip(roi))

# 6. Fetch Labels (y) - Dynamic World
dw_image = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
            .filterBounds(roi)
            .filterDate('2022-01-01', '2022-12-31')
            .select('trees')
            .mean()
            .clip(roi))
veg_mask = dw_image.gt(0.5) # Convert to strict 1 (Veg) or 0 (No Veg)

# 7. Export to Google Drive
ee.batch.Export.image.toDrive(
    image=s2_image, description='Sentinel2_Input_X', folder='Carbon_ML_Dataset', scale=10, region=roi, fileFormat='GeoTIFF'
).start()

ee.batch.Export.image.toDrive(
    image=veg_mask, description='Vegetation_Label_y', folder='Carbon_ML_Dataset', scale=10, region=roi, fileFormat='GeoTIFF'
).start()

print("Export tasks sent! Wait 5-10 minutes for them to appear in your Google Drive before running the next cell.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Export tasks sent! Wait 5-10 minutes for them to appear in your Google Drive before running the next cell.


In [35]:
import rasterio
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import segmentation_models_pytorch as smp
from google.colab import drive
import os

# Ensure Drive is mounted (in case your session restarted)
drive.mount('/content/drive')

# 1. Define Paths
x_path = '/content/drive/MyDrive/Carbon_ML_Dataset/Sentinel2_Input_X.tif'
y_path = '/content/drive/MyDrive/Carbon_ML_Dataset/Vegetation_Label_y.tif'

# Sanity check to make sure the files exist before running heavy math
if not os.path.exists(x_path) or not os.path.exists(y_path):
    raise FileNotFoundError("Could not find the .tif files! Please check your Google Drive folder.")

print("Files found! Slicing images into 256x256 patches...")

# 2. Load and Slice Data
patch_size = 256
x_patches, y_patches = [], []

with rasterio.open(x_path) as src_x, rasterio.open(y_path) as src_y:
    x_img, y_img = src_x.read(), src_y.read()
    _, h, w = x_img.shape

    for i in range(0, h - patch_size, patch_size):
        for j in range(0, w - patch_size, patch_size):
            x_patch = x_img[:, i:i+patch_size, j:j+patch_size]
            y_patch = y_img[:, i:i+patch_size, j:j+patch_size]

            # Only keep patches that actually have image data (ignoring empty black borders)
            if not np.all(x_patch == 0):
                x_patches.append(x_patch)
                y_patches.append(y_patch)

print(f"Created {len(x_patches)} valid patches for training!")

# Convert lists to PyTorch Tensors
X_tensor = torch.tensor(np.array(x_patches), dtype=torch.float32)
y_tensor = torch.tensor(np.array(y_patches), dtype=torch.float32)

# Create a DataLoader to feed the model in batches of 16 images at a time
dataloader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=16, shuffle=True)

# 3. Build the U-Net Model
# We use Google Colab's GPU ("cuda") if it's available, otherwise standard CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Building model and sending it to: {device}")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation='sigmoid'
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Train the Model
epochs = 5 # Keeping it at 5 for a quick proof-of-concept run
print("Starting training loop...")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Average Loss: {epoch_loss/len(dataloader):.4f}")

# 5. Save the Model Weights
model_save_path = '/content/drive/MyDrive/Carbon_ML_Dataset/unet_vegetation_weights.pth'
torch.save(model.state_dict(), model_save_path)
print(f"✅ Training complete! Model successfully saved to: {model_save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files found! Slicing images into 256x256 patches...
Created 441 valid patches for training!
Building model and sending it to: cpu
Starting training loop...
Epoch 1/5 | Average Loss: 0.2028
Epoch 2/5 | Average Loss: 0.0904
Epoch 3/5 | Average Loss: 0.0664
Epoch 4/5 | Average Loss: 0.0503
Epoch 5/5 | Average Loss: 0.0505
✅ Training complete! Model successfully saved to: /content/drive/MyDrive/Carbon_ML_Dataset/unet_vegetation_weights.pth


In [36]:
#For checking status of cell 3. Continuously run this
import ee
import os

# 1. Check Earth Engine Task Status
print("--- Checking Earth Engine Servers ---")
try:
    tasks = ee.batch.Task.list()
    for task in tasks[:2]: # Check the two most recent tasks
        print(f"Task: {task.config.get('description', 'Unknown')}")
        print(f"Status: {task.status().get('state', 'Unknown')}")
except Exception as e:
    print("Could not fetch EE tasks. Did you run Cell 2 first?")

print("\n--- Checking Google Drive ---")
# 2. Check what Colab actually sees in your Drive
folder_path = '/content/drive/MyDrive/Carbon_ML_Dataset/'

if os.path.exists(folder_path):
    print("Folder found! Here is what is inside:")
    files = os.listdir(folder_path)
    if len(files) == 0:
        print(" -> Folder is empty. Still waiting on Earth Engine.")
    else:
        for f in files:
            print(f" -> {f}")
else:
    print(f"ERROR: The folder {folder_path} does not exist in Colab's view.")
    print("Did you click 'Allow' when Colab asked to connect to your Google Drive?")

--- Checking Earth Engine Servers ---
Task: Vegetation_Label_y
Status: COMPLETED
Task: Sentinel2_Input_X
Status: COMPLETED

--- Checking Google Drive ---
Folder found! Here is what is inside:
 -> Sentinel2_Input_X.tif
 -> Vegetation_Label_y.tif
 -> unet_vegetation_weights.pth


In [37]:
import numpy as np

def calculate_avle_and_carbon(mask_t1, mask_t2, prob_t2, region_weight=1.2, emission_factor=150):
    """
    Core logic for calculating carbon loss from deforestation.
    """
    # 1. Change Detection: Loss = V_t1 - V_t2
    # Finds pixels that were trees in t1, but are NOT trees in t2
    loss_mask = (mask_t1 == 1) & (mask_t2 == 0)

    # 2. AVLE Score
    # Confidence is how sure the model is that the trees are gone (1.0 - probability of vegetation)
    confidence = 1.0 - prob_t2[loss_mask]
    avle_score = np.sum(confidence * region_weight)

    # 3. Carbon Estimation
    # 1 pixel = 10m x 10m = 100 sq meters = 0.01 hectares
    area_hectares = np.sum(loss_mask) * 0.01
    carbon_released = area_hectares * emission_factor

    return area_hectares, avle_score, carbon_released

# --- Mock Data to Test the Math ---
# Simulating a 256x256 patch of the Amazon
mask_2022 = np.ones((256, 256)) # 100% forest in 2022
mask_2023 = np.ones((256, 256))
mask_2023[100:150, 100:150] = 0 # Cut down a 50x50 square of trees in 2023

# Simulating the AI's confidence
prob_2023 = np.full((256, 256), 0.9) # AI is 90% sure the rest is still forest
prob_2023[100:150, 100:150] = 0.1 # AI is only 10% sure there are trees here (meaning 90% sure it's dirt)

print("Running AVLE and Carbon Math Engine...\n")
area, score, carbon = calculate_avle_and_carbon(mask_2022, mask_2023, prob_2023)

print(f"Area Lost: {area:.2f} Hectares")
print(f"AVLE Score: {score:.2f}")
print(f"Carbon Released: {carbon:.2f} Tons CO2")

Running AVLE and Carbon Math Engine...

Area Lost: 25.00 Hectares
AVLE Score: 2700.00
Carbon Released: 3750.00 Tons CO2
